# 🛰️ ANTI-UAV RGB-ONLY: TẠO BỘ DỮ LIỆU SIÊU TỐC
### Google Colab Notebook: Lọc & Khử Triệt Để Hồng Ngoại (IR) Trực Tiếp Từ File `VOC_AntiUAV_stride3.tar` Có Sẵn

> **So Sánh Hai Phương Pháp**:
> * **Cách 1: Can thiệp trực tiếp vào `VOC_AntiUAV_stride3.tar` (KHUYÊN DÙNG)**:
>   * Các frame Visible đã được trích xuất sẵn, chất lượng cao và đồng bộ.
>   * Chỉ cần lọc bỏ các file `*_ir_*` và tự động sinh file Triplets CSV.
>   * ⏱️ **Thời gian: Chỉ mất ~3 - 5 phút** (Nhanh gấp 10 - 15 lần so với làm lại từ đầu)!
> * **Cách 2: Trích xuất lại từ các video MP4 + JSON gốc**:
>   * Phải đọc và decode lại hàng trăm video qua Google Drive I/O.
>   * ⏱️ **Thời gian: Mất ~35 - 60 phút**.


In [ ]:
from google.colab import drive
import os
import sys
import shutil
import glob
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

print("=" * 70)
print(" 1. KẾT NỐI GOOGLE DRIVE")
print("=" * 70)

drive.mount('/content/drive')

TAR_PATH = "/content/drive/MyDrive/Anti-UAV-RGBT/VOC_AntiUAV_stride3.tar"
if os.path.exists(TAR_PATH):
    size_gb = os.path.getsize(TAR_PATH) / (1024 ** 3)
    print(f">> [OK] Đã tìm thấy file TAR có sẵn: {TAR_PATH} ({size_gb:.2f} GB)")
else:
    print(f">> [CẢNH BÁO] Chưa thấy file tại: {TAR_PATH}")
    # Tìm kiếm các file .tar khác nếu có
    found = glob.glob("/content/drive/MyDrive/**/VOC_*.tar", recursive=True)
    if found:
        print(f"   Gợi ý file tìm thấy: {found}")


In [ ]:
#@title ⚙️ 2. CẤU HÌNH ĐƯỜNG DẪN & ĐẦU RA
#@markdown Thiết lập đường dẫn file tar đầu vào và thư mục xuất:

INPUT_TAR_PATH = "/content/drive/MyDrive/Anti-UAV-RGBT/VOC_AntiUAV_stride3.tar" #@param {type:"string"}
OUTPUT_RGB_DIR = "/content/VOC_AntiUAV_RGB" #@param {type:"string"}
EXPORT_TAR_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_TAR = "/content/drive/MyDrive/Anti-UAV-RGBT/VOC_AntiUAV_RGB_stride3.tar" #@param {type:"string"}

print(f">> File TAR nguồn : {INPUT_TAR_PATH}")
print(f">> Thư mục trích  : {OUTPUT_RGB_DIR}")
print(f">> Lưu về Drive   : {EXPORT_TAR_TO_DRIVE} -> {DRIVE_OUTPUT_TAR}")


---
## PHƯƠNG PHÁP 1: GIẢI NÉN CHỌN LỌC (CHỈ LẤY VISIBLE, BỎ HOÀN TOÀN IR)
* Dùng lệnh `tar` với bộ lọc `--exclude='*_ir_*'` để bỏ qua $100\%$ các file hồng ngoại ngay trong quá trình giải nén.
* Tiết kiệm ngay $50\%$ dung lượng ổ đĩa và hoàn tất chỉ trong 1-2 phút!


In [ ]:
import subprocess
import time

print("=" * 70)
print(" BẮT ĐẦU GIẢI NÉN CHỌN LỌC (LOẠI BỎ TOÀN BỘ FILE HỒNG NGOẠI IR)")
print("=" * 70)

t0 = time.time()
os.makedirs("/content/temp_extract", exist_ok=True)

# Giải nén bỏ qua toàn bộ *_ir_* ngay từ đầu
cmd = f"tar -xf \"{INPUT_TAR_PATH}\" --exclude='*_ir_*' -C /content/temp_extract"
print(f">> Đang thực thi: {cmd}")
subprocess.run(cmd, shell=True, check=True)

# Tìm thư mục gốc được bung ra bên trong temp_extract
extracted_dirs = [os.path.join("/content/temp_extract", d) for d in os.listdir("/content/temp_extract") if os.path.isdir(os.path.join("/content/temp_extract", d))]
if len(extracted_dirs) == 1 and ("VOC" in extracted_dirs[0] or "JPEGImages" in os.listdir(extracted_dirs[0])):
    source_extracted = extracted_dirs[0]
else:
    source_extracted = "/content/temp_extract"

if os.path.exists(OUTPUT_RGB_DIR):
    shutil.rmtree(OUTPUT_RGB_DIR)
shutil.move(source_extracted, OUTPUT_RGB_DIR)
shutil.rmtree("/content/temp_extract", ignore_errors=True)

# Xóa vét lại lần nữa để đảm bảo không còn bất kỳ file _ir_ nào
!find "{OUTPUT_RGB_DIR}/JPEGImages" -name "*_ir_*" -delete 2>/dev/null || true
!find "{OUTPUT_RGB_DIR}/Annotations" -name "*_ir_*" -delete 2>/dev/null || true

n_jpg = len(glob.glob(f"{OUTPUT_RGB_DIR}/JPEGImages/*.jpg"))
n_xml = len(glob.glob(f"{OUTPUT_RGB_DIR}/Annotations/*.xml"))

elapsed = time.time() - t0
print(f"\n✅ GIẢI NÉN & LỌC SẠCH HOÀN TẤT TRONG {elapsed:.1f} GIÂY!")
print(f"   + Tổng số ảnh Visible JPG  : {n_jpg}")
print(f"   + Tổng số nhãn Visible XML : {n_xml}")
print(f"   + Thư mục đích             : {OUTPUT_RGB_DIR}")


---
## PHẦN 2: LÀM SẠCH IMAGESETS & TỰ ĐỘNG SINH TRIPLETS RGB-ONLY
* Lọc sạch file `ImageSets/Main/*.txt` (loại bỏ mọi frame IR).
* Tự động tạo `train_triplets.csv`, `val_triplets.csv`, `test_triplets.csv` trực tiếp từ `Annotations/*.xml` (chuẩn hóa $t-1, t, t+1$, tính vận tốc, kích thước drone) chỉ mất ~3-5 giây!


In [ ]:
import xml.etree.ElementTree as ET

print("=" * 70)
print(" LÀM SẠCH DANH SÁCH IMAGESETS & CẬP NHẬT/SINH TRIPLETS CSV")
print("=" * 70)

# 1. Làm sạch ImageSets/Main/*.txt (loại bỏ mọi frame IR)
sets_dir = os.path.join(OUTPUT_RGB_DIR, "ImageSets", "Main")
split_map = {}
if os.path.exists(sets_dir):
    for txt_file in os.listdir(sets_dir):
        if txt_file.endswith(".txt"):
            p = os.path.join(sets_dir, txt_file)
            with open(p, "r", encoding="utf-8") as f:
                lines = [l.strip() for l in f if l.strip() and "_ir_" not in l]
            with open(p, "w", encoding="utf-8") as f:
                f.write("\n".join(lines))
            sp_name = txt_file.replace(".txt", "")
            for stem in lines:
                split_map[stem] = sp_name
            print(f">> [x] Đã lọc sạch: {txt_file} ({len(lines)} frames visible)")

# 2. Xử lý Triplets CSV:
trip_train_path = os.path.join(OUTPUT_RGB_DIR, "train_triplets.csv")
trip_val_path   = os.path.join(OUTPUT_RGB_DIR, "val_triplets.csv")
trip_test_path  = os.path.join(OUTPUT_RGB_DIR, "test_triplets.csv")

existing_triplets = [p for p in [trip_train_path, trip_val_path, trip_test_path] if os.path.exists(p)]

if len(existing_triplets) > 0:
    for p_csv in existing_triplets:
        df_trip = pd.read_csv(p_csv)
        if "vis_t" in df_trip.columns:
            df_trip["ir_tm1"] = df_trip["vis_tm1"]
            df_trip["ir_t"]   = df_trip["vis_t"]
            df_trip["ir_tp1"] = df_trip["vis_tp1"]
            df_trip.to_csv(p_csv, index=False)
            print(f">> [x] Đã đồng bộ chuẩn RGB-Only cho: {os.path.basename(p_csv)} ({len(df_trip)} mẫu)")
else:
    print(">> Chưa có sẵn file triplets CSV. Đang tự động quét Annotations/*.xml để sinh triplets siêu tốc...")
    xml_dir = os.path.join(OUTPUT_RGB_DIR, "Annotations")
    manifest_csv = os.path.join(OUTPUT_RGB_DIR, "paired_frames_manifest.csv")
    records = []
    
    if os.path.exists(manifest_csv):
        print(f">> Tìm thấy manifest: {manifest_csv}")
        mdf = pd.read_csv(manifest_csv)
        if "vis_stem" in mdf.columns:
            for _, row in mdf.iterrows():
                bbox_str = str(row.get("vis_bbox", ""))
                xmin, ymin, xmax, ymax = 0, 0, 0, 0
                if row.get("vis_exist", 0) == 1 and bbox_str and bbox_str != "[]":
                    try:
                        import ast
                        b = ast.literal_eval(bbox_str)
                        if len(b) == 4: xmin, ymin, xmax, ymax = b
                    except Exception: pass
                records.append({
                    "split": row.get("split", "train"),
                    "sequence_id": row.get("sequence_id", ""),
                    "frame_idx": int(row.get("frame_idx", 0)),
                    "stem": row["vis_stem"],
                    "exist": int(row.get("vis_exist", 0)),
                    "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax
                })

    if len(records) == 0 and os.path.exists(xml_dir):
        xml_files = sorted(glob.glob(os.path.join(xml_dir, "*.xml")))
        print(f">> Đang phân tích {len(xml_files)} file nhãn XML...")
        for xf in xml_files:
            fname = os.path.basename(xf)
            if "_ir_" in fname:
                continue
            stem = fname.replace(".xml", "")
            try:
                tree = ET.parse(xf)
                root = tree.getroot()
                seq_elem = root.find("source/sequence")
                seq_name = seq_elem.text if seq_elem is not None else ""
                f_elem = root.find("source/frame_id")
                frame_idx = int(f_elem.text) if f_elem is not None else 0
                ex_elem = root.find("source/exist")
                exist = int(ex_elem.text) if ex_elem is not None else 0
                
                xmin, ymin, xmax, ymax = 0, 0, 0, 0
                obj = root.find("object")
                if obj is not None and exist == 1:
                    b = obj.find("bndbox")
                    xmin = int(b.find("xmin").text)
                    ymin = int(b.find("ymin").text)
                    xmax = int(b.find("xmax").text)
                    ymax = int(b.find("ymax").text)
                
                split = split_map.get(stem, "")
                if not split:
                    if seq_name.startswith("val"): split = "val"
                    elif seq_name.startswith("test"): split = "test"
                    else: split = "train"
                
                records.append({
                    "split": split,
                    "sequence_id": seq_name,
                    "frame_idx": frame_idx,
                    "stem": stem,
                    "exist": exist,
                    "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax
                })
            except Exception:
                pass

    if len(records) > 0:
        df = pd.DataFrame(records)
        df["box_w"] = df["xmax"] - df["xmin"]
        df["box_h"] = df["ymax"] - df["ymin"]
        df["box_area"] = df["box_w"] * df["box_h"]
        df["cx"] = df["xmin"] + df["box_w"] / 2.0
        df["cy"] = df["ymin"] + df["box_h"] / 2.0
        
        df.sort_values(by=["sequence_id", "frame_idx"], inplace=True)
        df.reset_index(drop=True, inplace=True)
        
        df["prev_cx"] = df.groupby("sequence_id")["cx"].shift(1)
        df["prev_cy"] = df.groupby("sequence_id")["cy"].shift(1)
        df["prev_exist"] = df.groupby("sequence_id")["exist"].shift(1).fillna(0)
        disp = np.sqrt((df["cx"] - df["prev_cx"])**2 + (df["cy"] - df["prev_cy"])**2)
        df["velocity"] = np.where((df["exist"] == 1) & (df["prev_exist"] == 1), disp, 0.0).round(2)
        
        def get_motion_level(v):
            if v == 0: return "stationary"
            elif v <= 5: return "slow"
            elif v <= 15: return "medium"
            else: return "fast"
        df["motion_level"] = df["velocity"].apply(get_motion_level)

        def get_size_category(area):
            if area == 0: return "none"
            elif area < (32 * 32): return "tiny"
            elif area < (64 * 64): return "small"
            elif area < (96 * 96): return "medium"
            else: return "large"
        df["size_category"] = df["box_area"].apply(get_size_category)
        
        for sp in ["train", "val", "test"]:
            sub = df[df["split"] == sp]
            if len(sub) == 0: continue
            triplets = []
            for seq_id, group in sub.groupby("sequence_id"):
                group = group.sort_values("frame_idx").reset_index(drop=True)
                n = len(group)
                for i in range(n):
                    rt = group.iloc[i]
                    rtm1 = group.iloc[max(0, i - 1)]
                    rtp1 = group.iloc[min(n - 1, i + 1)]
                    triplets.append({
                        "split": sp,
                        "sequence_id": seq_id,
                        "frame_idx": rt["frame_idx"],
                        "vis_tm1": f"JPEGImages/{rtm1['stem']}.jpg",
                        "vis_t": f"JPEGImages/{rt['stem']}.jpg",
                        "vis_tp1": f"JPEGImages/{rtp1['stem']}.jpg",
                        "ir_tm1": f"JPEGImages/{rtm1['stem']}.jpg",
                        "ir_t": f"JPEGImages/{rt['stem']}.jpg",
                        "ir_tp1": f"JPEGImages/{rtp1['stem']}.jpg",
                        "xmin": rt["xmin"], "ymin": rt["ymin"],
                        "xmax": rt["xmax"], "ymax": rt["ymax"],
                        "box_w": rt["box_w"], "box_h": rt["box_h"],
                        "box_area": rt["box_area"],
                        "exist": rt["exist"],
                        "velocity": rt["velocity"],
                        "motion_level": rt["motion_level"],
                        "size_category": rt["size_category"]
                    })
            tdf = pd.DataFrame(triplets)
            out_p = os.path.join(OUTPUT_RGB_DIR, f"{sp}_triplets.csv")
            tdf.to_csv(out_p, index=False)
            print(f">> [x] Đã tạo thành công: {os.path.basename(out_p)} ({len(tdf)} triplets)")
    else:
        print(">> [Cảnh báo] Không tìm thấy dữ liệu để tạo triplets!")

print("\n✅ TOÀN BỘ DỮ LIỆU ĐÃ ĐƯỢC CHUẨN HÓA THÀNH RGB-ONLY 100%!")


---
## PHẦN 3: KIỂM TRA TRỰC QUAN ẢNH VISIBLE & BOUNDING BOX
* Trích xuất ngẫu nhiên 4 mẫu trong tập triplets (`test_triplets.csv`, `val_triplets.csv` hoặc `train_triplets.csv`) để xác nhận tọa độ bounding box ôm khít lấy thân drone.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Tự động chọn file triplets có sẵn
chk_csv = None
for cand in [trip_test_path, trip_val_path, trip_train_path]:
    if cand and os.path.exists(cand):
        chk_csv = cand
        break

if chk_csv is None or not os.path.exists(chk_csv):
    raise FileNotFoundError("Chưa tìm thấy file triplets nào! Vui lòng kiểm tra và chạy lại Cell 6.")

print(f">> Đang kiểm tra trực quan từ file: {os.path.basename(chk_csv)}")
df_chk = pd.read_csv(chk_csv)
pos_chk = df_chk[df_chk["exist"] == 1].sample(min(4, len(df_chk[df_chk["exist"] == 1])), random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=150)
axes = axes.flatten()

for i, row in pos_chk.iterrows():
    ax = axes[i]
    rel_path = row["vis_t"]
    full_path = os.path.join(OUTPUT_RGB_DIR, rel_path)
    im = cv2.cvtColor(cv2.imread(full_path), cv2.COLOR_BGR2RGB)
    ax.imshow(im)
    
    bw = row["xmax"] - row["xmin"]
    bh = row["ymax"] - row["ymin"]
    ax.set_title(f"{row['sequence_id']} | Frame {row['frame_idx']} | Drone BBox: {bw:.0f}x{bh:.0f}px", fontsize=11, fontweight="bold")
    ax.axis("off")

    rect = patches.Rectangle(
        (row["xmin"], row["ymin"]), bw, bh,
        linewidth=2.5, edgecolor="#00FF00", facecolor="none", linestyle="--"
    )
    ax.add_patch(rect)
    ax.text(row["xmin"], max(0, row["ymin"] - 10), f"Drone GT ({bw:.0f}x{bh:.0f})", color="white", fontsize=9, fontweight="bold",
            bbox=dict(facecolor="#00AA00", edgecolor="none", pad=2))

plt.tight_layout()
preview_p = os.path.join(OUTPUT_RGB_DIR, "preview_rgb_only_check.png")
plt.savefig(preview_p, bbox_inches="tight")
plt.show()
print(f">> [x] Đã lưu ảnh kiểm tra tại: {preview_p}")


---
## PHẦN 4: ĐÓNG GÓI THÀNH FILE `.TAR` VÀ LƯU VỀ GOOGLE DRIVE
* Đóng gói thư mục `VOC_AntiUAV_RGB` thành file `.tar` siêu nhẹ (dung lượng giảm hơn một nửa).
* Lưu thẳng về Google Drive: `/content/drive/MyDrive/Anti-UAV-RGBT/VOC_AntiUAV_RGB_stride3.tar`.


In [ ]:
print("=" * 70)
print(" ĐÓNG GÓI DATASET RGB-ONLY THÀNH FILE .TAR")
print("=" * 70)

tar_local = "/content/VOC_AntiUAV_RGB_stride3.tar"

parent_dir = os.path.dirname(OUTPUT_RGB_DIR)
base_name = os.path.basename(OUTPUT_RGB_DIR)

t0 = time.time()
print(f">> Đang tạo file TAR: {tar_local} ...")
cmd = f"tar -cf {tar_local} -C {parent_dir} {base_name}"
subprocess.run(cmd, shell=True, check=True)

size_gb = os.path.getsize(tar_local) / (1024 ** 3)
elapsed = time.time() - t0
print(f"✅ ĐÃ ĐÓNG GÓI XONG TRONG {elapsed:.1f}s | Dung lượng: {size_gb:.2f} GB (Đã giảm hơn 50% so với bản cũ)!")

if EXPORT_TAR_TO_DRIVE:
    print(f"\n>> Đang sao lưu file về Google Drive tại: {DRIVE_OUTPUT_TAR} ...")
    os.makedirs(os.path.dirname(DRIVE_OUTPUT_TAR), exist_ok=True)
    shutil.copyfile(tar_local, DRIVE_OUTPUT_TAR)
    print(f"✅ ĐÃ SAO LƯU THÀNH CÔNG VỀ GOOGLE DRIVE!")
    print(f"   Đường dẫn: {DRIVE_OUTPUT_TAR}")

print("\n" + "=" * 70)
print(" HOÀN THÀNH ĐÓNG GÓI TAR!")
print("=" * 70)


---
## PHẦN 5: TẢI TRỰC TIẾP DATASET LÊN KAGGLE (PUSH TO KAGGLE DATASET)
* Tự động xác thực Kaggle API (qua Form điền `KAGGLE_USERNAME`/`KAGGLE_KEY`, file `kaggle.json` trên Drive, hoặc tải file lên trực tiếp).
* Tự động tạo mới hoặc cập nhật phiên bản dataset trên Kaggle (`kaggle datasets create` / `kaggle datasets version`).
* Xuất đường dẫn trực tiếp để bạn có thể gắn vào Kaggle Notebook và bắt đầu huấn luyện ngay!


In [ ]:
#@title 🚀 5. TẢI DATASET LÊN KAGGLE (KAGGLE DATASET UPLOAD)
#@markdown **Cấu hình thông tin Dataset trên Kaggle:**

KAGGLE_DATASET_SLUG = "anti-uav-rgb-stride3" #@param {type:"string"}
KAGGLE_DATASET_TITLE = "Anti UAV RGB Stride 3 Dataset" #@param {type:"string"}
IS_PUBLIC = False #@param {type:"boolean"}

#@markdown **Xác thực tài khoản Kaggle:**
#@markdown *(Nếu để trống 2 ô dưới, Colab sẽ tự tìm file kaggle.json trên Google Drive hoặc hiện nút Upload từ máy tính)*
KAGGLE_USERNAME = "" #@param {type:"string"}
KAGGLE_KEY = "" #@param {type:"string"}

import os
import json
import shutil
import subprocess
import glob

print("=" * 70)
print(" 🚀 BẮT ĐẦU TẢI BỘ DỮ LIỆU LÊN KAGGLE")
print("=" * 70)

# Cài đặt / Cập nhật Kaggle CLI
subprocess.run("pip install -q --upgrade kaggle", shell=True, check=True)

# 1. Cấu hình credentials Kaggle (Hỗ trợ cả chuẩn Access Token mới KGAT_ và chuẩn cũ)
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_target = os.path.join(kaggle_dir, "kaggle.json")
access_token_target = os.path.join(kaggle_dir, "access_token")

token_val = KAGGLE_KEY.strip()
user_val = KAGGLE_USERNAME.strip()

if not token_val:
    # Tìm kiếm các vị trí có thể có kaggle.json hoặc access_token
    candidates = [
        access_token_target,
        "/content/kaggle.json",
        "/content/drive/MyDrive/kaggle.json",
        "/content/drive/MyDrive/Anti-UAV-RGBT/kaggle.json",
        kaggle_json_target
    ]
    for cand in candidates:
        if os.path.exists(cand) and os.path.getsize(cand) > 0:
            if cand.endswith("access_token"):
                with open(cand, "r", encoding="utf-8") as f:
                    token_val = f.read().strip()
                print(f">> [x] Đã tìm thấy access_token tại: {cand}")
                break
            else:
                try:
                    with open(cand, "r", encoding="utf-8") as f:
                        cred_data = json.load(f)
                        token_val = cred_data.get("key", "") or cred_data.get("token", "")
                        if not user_val:
                            user_val = cred_data.get("username", "")
                    print(f">> [x] Đã tìm thấy kaggle.json tại: {cand}")
                    break
                except Exception:
                    pass

    if not token_val:
        print(">> [!] Chưa có cấu hình xác thực Kaggle.")
        print(">> Vui lòng tải file kaggle.json lên (Lấy tại kaggle.com -> Settings -> Create New Token):")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith(".json"):
                try:
                    with open(fn, "r", encoding="utf-8") as f:
                        cred_data = json.load(f)
                        token_val = cred_data.get("key", "") or cred_data.get("token", "")
                        if not user_val:
                            user_val = cred_data.get("username", "")
                    print(f">> [x] Đã nạp thành công token từ: {fn}")
                    break
                except Exception:
                    pass

if not token_val:
    raise FileNotFoundError("Chưa có API Token Kaggle! Vui lòng điền KAGGLE_KEY hoặc tải file kaggle.json.")

# BẮT BUỘC: Lưu token vào ~/.kaggle/access_token (Dành cho token chuẩn mới KGAT_...)
with open(access_token_target, "w", encoding="utf-8") as f:
    f.write(token_val)
os.chmod(access_token_target, 0o600)

# Đồng thời lưu ~/.kaggle/kaggle.json (Tương thích ngược với các phiên bản CLI)
with open(kaggle_json_target, "w", encoding="utf-8") as f:
    json.dump({"username": user_val, "key": token_val}, f)
os.chmod(kaggle_json_target, 0o600)

# Thiết lập toàn bộ các biến môi trường
os.environ["KAGGLE_API_TOKEN"] = token_val
os.environ["KAGGLE_KEY"] = token_val
if user_val:
    os.environ["KAGGLE_USERNAME"] = user_val

kaggle_user = user_val or "my-account"

# Kiểm tra xác thực ngay lập tức
print(">> Đang kiểm tra xác thực tài khoản Kaggle...")
auth_check = subprocess.run("kaggle datasets list -m --page 1", shell=True, capture_output=True, text=True)
if "Authentication required" in auth_check.stderr or "Unauthorized" in auth_check.stderr:
    print(">> [CẢNH BÁO] Kiểm tra lại token! Chi tiết:", auth_check.stderr.strip())
    raise RuntimeError("Kaggle API Authentication failed! Vui lòng kiểm tra lại Token.")
else:
    print(f">> [OK] Xác thực Kaggle API thành công cho tài khoản: @{kaggle_user}")

# 2. Chuẩn bị thư mục Staging cho Kaggle Upload
staging_dir = "/content/kaggle_dataset_staging"
if os.path.exists(staging_dir):
    shutil.rmtree(staging_dir)
os.makedirs(staging_dir, exist_ok=True)

# Xác định file tar nguồn
tar_file = "/content/VOC_AntiUAV_RGB_stride3.tar"
if not os.path.exists(tar_file) and "DRIVE_OUTPUT_TAR" in globals() and os.path.exists(DRIVE_OUTPUT_TAR):
    tar_file = DRIVE_OUTPUT_TAR

if not os.path.exists(tar_file):
    raise FileNotFoundError(f"Không tìm thấy file tar tại {tar_file}. Vui lòng chạy Cell 10 trước!")

tar_dest = os.path.join(staging_dir, "VOC_AntiUAV_RGB_stride3.tar")
size_gb = os.path.getsize(tar_file) / (1024 ** 3)
print(f">> Chuẩn bị đóng gói file TAR ({size_gb:.2f} GB)...")

try:
    os.link(tar_file, tar_dest) # Nhanh tức thì, không tốn thêm dung lượng đĩa
except Exception:
    shutil.copyfile(tar_file, tar_dest)

# Đồng thời copy 3 file CSV triplets vào staging để Kaggle hiển thị bảng xem trước (preview) trên web
for csv_name in ["train_triplets.csv", "val_triplets.csv", "test_triplets.csv"]:
    src_csv = os.path.join(OUTPUT_RGB_DIR, csv_name)
    if os.path.exists(src_csv):
        shutil.copyfile(src_csv, os.path.join(staging_dir, csv_name))

# 3. Tạo metadata dataset-metadata.json
dataset_id = f"{kaggle_user}/{KAGGLE_DATASET_SLUG}"
metadata = {
    "title": KAGGLE_DATASET_TITLE,
    "id": dataset_id,
    "licenses": [{"name": "CC0-1.0"}]
}
meta_path = os.path.join(staging_dir, "dataset-metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f">> Dataset Identifier: {dataset_id}")

# 4. Kiểm tra trạng thái dataset đã tồn tại hay chưa
check_res = subprocess.run(f"kaggle datasets status {dataset_id}", shell=True, capture_output=True, text=True)
is_existing = (check_res.returncode == 0 and "ready" in check_res.stdout.lower())

if not is_existing:
    print(f">> Dataset chưa có trên Kaggle. Đang tạo mới: {dataset_id} ...")
    pub_arg = "-u" if IS_PUBLIC else ""
    !kaggle datasets create -p "{staging_dir}" {pub_arg}
else:
    print(f">> Dataset đã tồn tại. Đang tải lên PHIÊN BẢN MỚI cho: {dataset_id} ...")
    !kaggle datasets version -p "{staging_dir}" -m "Update clean RGB-only stride 3 dataset"

print("\n" + "=" * 70)
print("🎉 BỘ DỮ LIỆU ĐÃ ĐƯỢC TẢI LÊN KAGGLE THÀNH CÔNG!")
print(f"👉 Link truy cập: https://www.kaggle.com/datasets/{dataset_id}")
print("=" * 70)
